### Dataset

In [1]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor
from torch.utils.data import random_split, DataLoader
import torch

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")

class RxHandBDDataset(Dataset):
    def __init__(self, csv_path, image_dir, processor):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]  # Get the row at index idx

        image_path = os.path.join(self.image_dir, row["Images"])
        image = Image.open(image_path).convert("RGB")

        text = row["Text"]

        # Converting our image pixels to tensor
        pixel_values = self.processor(
            images=image,
            return_tensors="pt"
        ).pixel_values.squeeze(0)

        # Convert our ground truth to token ids
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=64,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        # Dataloader automatically batches these so its fine
        return {
            "pixel_values": pixel_values,
            "labels": labels,
        }

In [2]:
dataset = RxHandBDDataset(
    "/workspace/rxhandbd/RxHandBD-ML/Train_Label.csv",
    "/workspace/rxhandbd/RxHandBD-ML/Train_Set",
    processor
)

val_fraction = 0.08

num_total = len(dataset)
num_val = int(num_total * val_fraction)
num_train = num_total - num_val

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    dataset,
    [num_train, num_val],
    generator=generator
)

print("train:", len(train_dataset))
print("val:", len(val_dataset))


train: 4106
val: 357


### DataLoader

In [3]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

### Align the Model to the Processor. 
Make sure that start, pad, and eos are the same

In [4]:
def align_model_to_processor(model, processor):
    tokenizer = processor.tokenizer

    model.config.decoder_start_token_id = tokenizer.cls_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.sep_token_id

    model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.sep_token_id

    # Useful for generation later
    model.generation_config.max_length = 64
    model.generation_config.num_beams = 1

    return model

### Define the VisionEncoderDecoder Model

In [5]:
import torch
from transformers import VisionEncoderDecoderModel

device = torch.device("cuda")

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-handwritten"
).to(device)

model = align_model_to_processor(model, processor)



Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Sanity check special token alignment

In [6]:
def check_trocr_alignment(model, processor):
    tokenizer = processor.tokenizer

    expected_decoder_start = tokenizer.cls_token_id
    expected_pad = tokenizer.pad_token_id
    expected_eos = tokenizer.sep_token_id

    checks = {
        "model.config.decoder_start_token_id": (
            model.config.decoder_start_token_id,
            expected_decoder_start,
        ),
        "model.config.pad_token_id": (
            model.config.pad_token_id,
            expected_pad,
        ),
        "model.config.eos_token_id": (
            model.config.eos_token_id,
            expected_eos,
        ),
        "model.generation_config.decoder_start_token_id": (
            model.generation_config.decoder_start_token_id,
            expected_decoder_start,
        ),
        "model.generation_config.pad_token_id": (
            model.generation_config.pad_token_id,
            expected_pad,
        ),
        "model.generation_config.eos_token_id": (
            model.generation_config.eos_token_id,
            expected_eos,
        ),
    }

    for name, (actual, expected) in checks.items():
        assert actual == expected, f"{name}: expected {expected}, got {actual}"

    print("TrOCR alignment check passed.")
    print(f"decoder_start_token_id = {expected_decoder_start}")
    print(f"pad_token_id           = {expected_pad}")
    print(f"eos_token_id           = {expected_eos}")

check_trocr_alignment(model, processor)

TrOCR alignment check passed.
decoder_start_token_id = 0
pad_token_id           = 1
eos_token_id           = 2


### Testing a few batches

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 3

for epoch in range(num_epochs):
    # --------------------
    # Train
    # --------------------
    model.train()

    total_train_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = {
            "pixel_values": batch["pixel_values"].to(device),
            "labels": batch["labels"].to(device),
        }

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_train_loss += loss.item()

        if step % 25 == 0:
            print(f"epoch {epoch+1}, step {step}, train loss = {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)

    # --------------------
    # Validation
    # --------------------
    model.eval()

    total_val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            batch = {
                "pixel_values": batch["pixel_values"].to(device),
                "labels": batch["labels"].to(device),
            }

            outputs = model(**batch)
            loss = outputs.loss

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    print(
        f"Epoch {epoch+1}/{num_epochs} finished | "
        f"train loss = {avg_train_loss:.4f} | "
        f"val loss = {avg_val_loss:.4f}"
    )

epoch 1, step 0, train loss = 12.2036
epoch 1, step 25, train loss = 2.2543
epoch 1, step 50, train loss = 3.7312
epoch 1, step 75, train loss = 0.8766
epoch 1, step 100, train loss = 0.7701
epoch 1, step 125, train loss = 1.8326
epoch 1, step 150, train loss = 2.1562
epoch 1, step 175, train loss = 2.2121
epoch 1, step 200, train loss = 1.2678
epoch 1, step 225, train loss = 1.7762
epoch 1, step 250, train loss = 0.9945
epoch 1, step 275, train loss = 2.0554
epoch 1, step 300, train loss = 0.8817
epoch 1, step 325, train loss = 2.3795
epoch 1, step 350, train loss = 1.4677
epoch 1, step 375, train loss = 2.0880
epoch 1, step 400, train loss = 2.1809
epoch 1, step 425, train loss = 1.2443
epoch 1, step 450, train loss = 0.9123
epoch 1, step 475, train loss = 1.0810
epoch 1, step 500, train loss = 2.9527
epoch 1, step 525, train loss = 1.6241
epoch 1, step 550, train loss = 0.8169
epoch 1, step 575, train loss = 0.2473
epoch 1, step 600, train loss = 1.8772
epoch 1, step 625, train loss

### Inference Check

In [10]:
model.eval()

batch = next(iter(val_loader))

pixel_values = batch["pixel_values"].to(device)

with torch.no_grad():
    generated_ids = model.generate(pixel_values)

pred_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

print(pred_texts)

labels = batch["labels"].clone()

# convert -100 back so we can decode labels
labels[labels == -100] = processor.tokenizer.pad_token_id

true_texts = processor.batch_decode(
    labels,
    skip_special_tokens=True
)

for pred, true in zip(pred_texts, true_texts):
    print("PRED:", pred)
    print("TRUE:", true)
    print()

['Neurolin', 'phllopen-D', 'Capsule', 'Tablet']
PRED: Neurolin
TRUE: Neurolin

PRED: phllopen-D
TRUE: Phylopen-Ds

PRED: Capsule
TRUE: Capsule

PRED: Tablet
TRUE: Tablet

